# N036 챕터08 실습 — 제목과 링크만 있는 상태에서 실제로 취재해보기

이 노트북은 [N036 뉴스레터 에이전트 만들기](../N036_뉴스레터에이전트(newsletter-agent).md) 강의 정리 노트의
**챕터 8 "제목과 링크만 있는 상태"**를 Colab에서 직접 실습하기 위한 노트북입니다.

## 이 노트북에서 하는 것
1. 실제 Hacker News 기사 3건을 가져와 본문을 추출하고, `headline`/`summary`/`why` 세 칸으로 나눠 LLM으로 취재해봅니다 (본문 실습)
2. LangGraph의 `Send`로 기사 수만큼 취재 노드를 동적으로 펼쳐봅니다 (본문 실습)
3. 🚀 더 해보기 1 — 한글 포함 여부를 코드로 검사하고 실패하면 재요청하는 장치를 붙여봅니다
4. 🚀 더 해보기 2 — 같은 기사를 temperature 0과 0.7로 각각 세 번씩 요약해 차이를 비교합니다

## 실행 순서
위에서부터 순서대로 셀을 실행하세요(Shift+Enter). "0. 환경 설정" 섹션을 먼저 반드시 실행해야 뒤 섹션이 동작합니다.


## 0. 환경 설정

### 0-1. 런타임
GPU가 필요 없는 실습입니다. 메뉴에서 **런타임 → 런타임 유형 변경 → CPU**로 두면 충분합니다.

### 0-2. 라이브러리 설치
아래 셀을 실행해 필요한 패키지를 설치합니다.


In [ ]:
!pip install -q openai langgraph pydantic trafilatura requests

### 0-3. API 키 설정 — Colab 보안 비밀 사용 (강의 방법 B, 권장)

**절대 키를 코드 셀에 문자열로 직접 적지 마세요.** N036 강의에서 배운 것처럼, 노트북에 키를 그대로
적어두면 실수로 공유하거나 GitHub에 올렸을 때 그대로 유출됩니다.

1. Colab 왼쪽 사이드바에서 **열쇠 모양 아이콘(🔑 보안 비밀)**을 클릭합니다.
2. **새 보안 비밀 추가**를 누르고 이름에 `OPENAI_API_KEY`, 값에 여러분의 키를 입력합니다.
3. **노트북 액세스** 토글을 켭니다(이 노트북에서만 읽을 수 있게 허용).
4. 아래 셀을 실행합니다 — 키 값이 화면에 출력되지 않고 환경변수로만 등록됩니다.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab 보안 비밀에서 키를 불러왔습니다.")
except Exception as e:
    # Colab이 아니거나 보안 비밀이 없는 경우: 로컬 .env 파일을 사용하세요.
    # (키를 이 셀에 직접 문자열로 적지 마세요!)
    try:
        from dotenv import load_dotenv
        load_dotenv()
        print(".env 파일에서 키를 불러왔습니다." if os.environ.get("OPENAI_API_KEY") else "키를 찾지 못했습니다.")
    except ImportError:
        print("키를 찾지 못했습니다:", e)

print("키 설정 여부:", "설정됨" if os.environ.get("OPENAI_API_KEY") else "비어 있음 — 위 안내를 따라 설정해주세요")

## 1. 실전 재료 확보 — 기사 3건의 제목·링크 가져오기

강의 본문(챕터 3~7)에서 RSS/API로 수집하고 관문·상대평가로 골라낸 "제목+링크뿐인" 상태를 재현하기 위해,
Hacker News의 Algolia 검색 API로 최근 AI 관련 기사 3건을 가져옵니다(공식 API라 인증이 필요 없습니다 —
챕터 3의 "사다리"에서 Hacker News는 API를 쓴다고 한 것과 같은 이유입니다).


In [ ]:
import requests

def fetch_candidate_articles(n=3):
    url = "https://hn.algolia.com/api/v1/search_by_date"
    params = {"tags": "story", "query": "AI", "hitsPerPage": 20}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    hits = r.json()["hits"]
    selected = []
    for h in hits:
        link = h.get("url")
        title = h.get("title")
        if not link or not title:
            continue  # Ask HN처럼 외부 링크가 없는 글은 취재 대상에서 제외 (G3 접근 관문과 같은 생각)
        selected.append({"title": title, "link": link})
        if len(selected) == n:
            break
    return selected

articles = fetch_candidate_articles(3)
for a in articles:
    print("-", a["title"], "|", a["link"])

## 2. 본문 추출

`trafilatura`로 각 링크의 본문을 뽑습니다. 챕터 4의 G1 관문(기준 600자)과 같은 생각으로,
본문이 너무 짧거나 추출에 실패하면 그 기사는 취재 대상에서 제외합니다 — 본문 없이 억지로
요약을 시키면 "재료가 없으면 언어 모델이 제목만 보고 요약을 지어낸다"는 문제가 그대로 재현되기 때문입니다.


In [ ]:
import trafilatura

G1_MIN_CHARS = 600

def extract_body(link):
    downloaded = trafilatura.fetch_url(link)
    if not downloaded:
        return None
    text = trafilatura.extract(downloaded)
    if not text or len(text) < G1_MIN_CHARS:
        return None
    return text

for a in articles:
    body = extract_body(a["link"])
    a["body"] = body
    status = f"{len(body)}자 확보" if body else "본문 확보 실패(G1 관문 탈락)"
    print(f"- {a['title'][:40]}... → {status}")

articles = [a for a in articles if a.get("body")]
print(f"\n최종 취재 대상: {len(articles)}건")

## 3. 출력 스키마 정의 — headline / summary / why

챕터 8 본문의 결론대로, 칸을 나누는 기준은 "무엇과 대조할 수 있는가"입니다.
`headline`·`summary`는 원문과 대조할 수 있고, `why`는 대조 대상이 없는 해석입니다.


In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

class NewsCard(BaseModel):
    headline: str = Field(description="카드에 들어갈 한국어 헤드라인. ~합니다체로 간결하게 작성합니다.")
    summary: str = Field(description="세 문장짜리 한국어 요약. 원문에 있는 사실만 담습니다.")
    why: str = Field(description="국내 개발팀에게 이 소식이 왜 중요한지 한 문장. 새로운 사실을 추가하지 말고 summary에 있는 내용만 근거로 삼습니다.")

SYSTEM_PROMPT = "당신은 국내 개발팀을 위한 AI 뉴스레터 기자입니다. 주어진 기사 본문만 근거로 삼아 답합니다."

def research_one(article, temperature=0.0, extra_instruction=None):
    user_prompt = f"""다음 기사를 바탕으로 headline, summary, why를 작성하세요.

제목: {article['title']}
본문:
{article['body'][:6000]}
"""
    if extra_instruction:
        user_prompt += f"\n\n{extra_instruction}"

    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=NewsCard,
    )
    return completion.choices[0].message.parsed

## 실습: 기사 세 건을 취재해 본다

강의 본문과 동일하게, 세 건을 하나씩 취재해 헤드라인과 요약을 확인합니다.
결과를 보면서 **언어가 한국어로 나왔는지**를 특히 눈여겨보세요 — 시스템 프롬프트와 스키마
설명 두 곳에 "한국어"라고 적었는데도, 원문이 영어인 기사는 영어로 응답이 돌아올 수 있습니다.


In [ ]:
import re

def has_korean(text):
    return bool(re.search(r'[가-힣]', text))

for a in articles:
    card = research_one(a)
    print("=" * 60)
    print("원문 제목:", a["title"])
    print("headline :", card.headline)
    print("summary  :", card.summary)
    print("why      :", card.why)
    print("한글 포함 여부(headline+summary):", has_korean(card.headline + card.summary))

### 객관식 퀴즈 (강의 원문)

> Q. 시스템 프롬프트에 '국내 개발팀을 위한'이라고 쓰고 스키마에도 '한국어'라고 명시했는데도
> 세 건 중 두 건이 영어로 나왔습니다. 가장 적절한 처방은?
> 1. 프롬프트에 '반드시 한국어로'를 대문자와 느낌표로 강조해서 다시 적는다
> 2. 출력에 한글이 포함됐는지 코드로 검사하고, 아니면 재요청하거나 번역 단계를 붙인다
> 3. 더 큰 모델로 바꾸면 지시를 더 잘 따르므로 모델을 교체한다
> 4. 영어 기사는 애초에 수집하지 않도록 소스 목록에서 제외한다

**정답: 2번.** 프롬프트는 요청이지 보장이 아니므로(1번), 코드로 확인 가능한 규칙(언어)은
반드시 코드로 강제해야 합니다. 3번은 근본 해결이 아니고(모델을 키워도 재현 가능), 4번은
문제(다국어 소스를 다뤄야 하는 요구 자체)를 회피할 뿐 해결하지 않습니다.


## 4. 기사 수만큼 펼치기 — LangGraph `Send`로 팬아웃

몇 건이 될지는 실행해봐야 압니다. `Send`를 쓰면 조건부 엣지가 리스트를 돌려주는 것만으로
그 개수만큼 같은 노드(`research`)를 동시에 띄울 수 있습니다. `drafted`에만 리듀서
(`operator.add`)를 붙여 두면, 워커들이 각자 반환한 한 건짜리 결과가 자동으로 합쳐집니다.


In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class GraphState(TypedDict):
    selected: list
    drafted: Annotated[list, operator.add]
    log: Annotated[list, operator.add]

def route_to_research(state: GraphState):
    # 조건부 엣지가 Send 리스트를 반환 → 개수만큼 research 노드를 동적으로 띄움
    return [Send("research", {"item": a}) for a in state["selected"]]

def research_node(payload):
    article = payload["item"]
    try:
        card = research_one(article)
    except Exception as e:
        return {"drafted": [], "log": [f"실패: {article['title'][:30]}... ({e})"]}
    return {"drafted": [card.model_dump()], "log": [f"완료: {article['title'][:30]}..."]}

builder = StateGraph(GraphState)
builder.add_node("research", research_node)
builder.add_conditional_edges(START, route_to_research, ["research"])
builder.add_edge("research", END)
graph = builder.compile()

result = graph.invoke({"selected": articles, "drafted": [], "log": []})

print(f"선택된 기사: {len(articles)}건 → 취재 완료: {len(result['drafted'])}건\n")
for line in result["log"]:
    print("-", line)
print()
for card in result["drafted"]:
    print("*", card["headline"])

워커(`research_node`)가 받는 것은 메인 State 전체가 아니라 `{"item": 기사 하나}`뿐입니다.
자기가 맡은 기사만 알고, 다른 기사가 몇 건인지도 모릅니다 — 그래서 워커 하나만 따로 떼어
테스트하기 쉽고, 한 건이 실패해도 나머지에 영향을 주지 않습니다(`research_node`의 `try/except`가
그 실패를 조용히 삼키지 않고 `log`에 남기는 것도 같은 이유입니다).


## 🚀 더 해보기 1 — 출력에 한글이 있는지 검사하고 없으면 한 번 더 요청하기

`re.search(r'[가-힣]', ...)`로 한글 포함 여부를 판정하고, 실패하면 "반드시 한국어로 다시
쓰세요"를 덧붙여 재요청합니다. 재시도 횟수를 세어 몇 건 중 몇 건에서 재시도가 일어났는지 기록합니다.

> ⚠️ **실제로 돌려본 결과, `headline + summary`를 합쳐서 검사하면 착시가 생깁니다.** `summary`만
> 한국어여도 `headline`이 영어 원문 그대로 남아있는 걸 가려버립니다(2건 중 1건이 그랬습니다).
> 그래서 아래 코드는 **`headline`과 `summary`를 각각 따로 검사**합니다 — 챕터 8이 "칸을 나누는
> 이유는 검증 방법이 다르기 때문"이라고 한 원칙을, 검사 코드 자체에도 적용한 것입니다.


In [ ]:
def research_one_with_korean_retry(article, max_retries=1):
    retries = 0
    card = research_one(article)
    while (not (has_korean(card.headline) and has_korean(card.summary))) and retries < max_retries:
        retries += 1
        card = research_one(
            article,
            extra_instruction="위 응답의 headline과 summary가 각각 한국어가 아니었습니다. 두 필드를 반드시 한국어로 다시 쓰세요.",
        )
    return card, retries

retry_count = 0
for a in articles:
    card, retries = research_one_with_korean_retry(a)
    field_check = f"headline={has_korean(card.headline)}, summary={has_korean(card.summary)}"
    retry_count += 1 if retries > 0 else 0
    print(f"- {a['title'][:40]}... | 재시도 {retries}회 | 최종: {field_check}")
    print(f"  headline: {card.headline}")

print(f"\n재시도가 발생한 기사: {retry_count} / {len(articles)}건")
print("→ 이 숫자가 '프롬프트만으로 될까'라는 질문에 대한 답입니다.")

## 🚀 더 해보기 2 — temperature 0 vs 0.7 비교

같은 기사를 temperature 0과 0.7로 각각 세 번씩 요약해, 문장 수·숫자·고유명사가 얼마나
달라지는지 봅니다. (완전한 형태소 분석 없이 정규식으로 근사한 지표입니다 — 정밀한 값이
아니라 "흔들리는 정도"를 보는 용도입니다.)


In [ ]:
def analyze_text(text):
    sentences = [s for s in re.split(r'[.!?。]', text) if s.strip()]
    numbers = re.findall(r'\d+(?:[.,]\d+)*', text)
    proper_nouns = re.findall(r'\b[A-Z][a-zA-Z]{2,}\b', text)  # 영문 고유명사 근사치
    return {
        "문장수": len(sentences),
        "숫자": sorted(set(numbers)),
        "영문고유명사": sorted(set(proper_nouns)),
    }

target_article = articles[0]
comparison = {}
for temp in [0.0, 0.7]:
    print(f"=== temperature={temp} ===")
    runs = []
    for i in range(3):
        card = research_one(target_article, temperature=temp)
        stats = analyze_text(card.summary)
        runs.append(stats)
        print(f"[{i+1}] {card.summary}")
        print(f"    → {stats}")
    comparison[temp] = runs
    print()

print("문장수 변동폭 (0.0):", {r['문장수'] for r in comparison[0.0]})
print("문장수 변동폭 (0.7):", {r['문장수'] for r in comparison[0.7]})
print("숫자 집합 변동 (0.0):", [r['숫자'] for r in comparison[0.0]])
print("숫자 집합 변동 (0.7):", [r['숫자'] for r in comparison[0.7]])

### 결과를 리포트에 기록하는 방법

이 노트북을 Colab에서 실행한 뒤, 각 셀의 **출력을 그대로 복사해서** Claude에게 붙여넣어
주시면 `실습/report.md`에 실제 실행 결과를 반영해 리포트를 완성합니다. 특히:
- 3번 섹션의 `headline`/`summary`/`why` 출력과 한글 포함 여부
- 더 해보기 1의 "재시도가 발생한 기사: N / M건" 줄
- 더 해보기 2의 문장수·숫자 변동폭 비교 출력

이 세 가지만 붙여주시면 충분합니다.
